<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/exercises/ex4/ex4_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML4NLP1
## Starting Point for Exercise 4

This notebook is the starting point for exercise 4. It has two parts:

1. **Part 1 (BERT):** fine-tune a pretrained BERT model for named entity recognition (NER, see [Section 18.3 in Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/18.pdf)) in a language of your choice, in four variants.
2. **Part 2 (GLiNER):** use GLiNER for zero-shot NER, compare it with your fine-tuned models on the same test set, and use it to anonymise your own texts.

Part 1 needs a GPU (Google Colab or JupyterHub); Part 2 also runs on a CPU.

Throughout the notebook, there are questions that you should address in your report. These are marked with 📝❓ and are collected again in the lab report section at the end of the notebook.

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

# Setup

In [ ]:
# Pin the library versions used in this exercise,
# so that your notebook runs the same way for you and for your reviewers.
%pip install -q transformers==5.16.1 datasets==5.1.0 accelerate==1.15.0 gliner==0.2.29 seqeval==1.2.2 sentencepiece protobuf

import torch
# Pick the best available device: NVIDIA GPU, then Apple Silicon GPU, then CPU
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

# Part 1: Named entity recognition with BERT

## Load and prepare the required data

In [ ]:
# Choose a supported language, apart from English (see the exercise description for the conditions).
# Examples: "de", "fr", "es", "it".
# NOTE: See dataset card for supported languages (https://huggingface.co/datasets/unimelb-nlp/wikiann)
chosen_language_code = ""

In [ ]:
import datasets

# NOTE: If the maximum sequence length exceeds the model's maximum
# sequence length, you need to make adjustments (for example, when
# choosing 'en')
test_set = datasets.load_dataset("unimelb-nlp/wikiann", chosen_language_code, split="test[:2000]")
train_set1000 = ...
train_set3000 = ...

**NOTE: Make sure that there are indeed as many data points in the above sets**

In [ ]:
print(train_set1000)
print(train_set3000)
print(test_set)

### Named entity recognition and the IOB format

In NER, the model assigns a tag to every word of a sentence. WikiANN has three entity types: persons (`PER`), organisations (`ORG`) and locations (`LOC`). Since an entity can span several words, the tags follow the IOB format (also called BIO): `B-` marks the first word of an entity, `I-` marks every following word of the same entity, and `O` marks words outside any entity. For example:

| Angela | Merkel | visited | the | University | of | Zurich |
|---|---|---|---|---|---|---|
| B-PER | I-PER | O | O | B-ORG | I-ORG | I-ORG |

📝❓ Why do we need separate `B-` and `I-` tags? What could not be represented with only `PER`, `ORG`, `LOC` and `O`?

In [ ]:
# WikiANN stores the tags as integers. This dictionary maps each tag name to its integer;
# you will need it to set the number of labels of your model and to convert
# the predicted integers back to tag names for the evaluation.
ner_tags = {tag: i for i, tag in enumerate(test_set.features["ner_tags"].feature.names)}
print(ner_tags)

**TODO: Inspect and Describe the Data, including Average and Maximum Input length (in tokens)**

📝❓Why do you need to be aware of the longest input length within your dataset? Which parameter of the model dictates this?

In [ ]:
from transformers import AutoTokenizer
import torch

# TODO: Load the tokenizer

In [ ]:
# TODO: Adjust by actually finding the maximum sequence length
max_sequence_length = 0

In [ ]:
print(max_sequence_length)

📝❓The dataset is split into words, and the assigned labels are for words. How should we deal with labels **after** tokenization? NOTE: Each word may be split into one or multiple tokens by the tokenizer.

In [ ]:
# TODO: Implement this function
def encode_and_align_labels(dataset, tokenizer, max_sequence_length):
    """Tokenizes the input tokens and aligns the word-level NER labels with the tokenized output."""

In [ ]:
# TODO: Encode the two training sets and the test set by applying the function above
encoded_test_set = ...
encoded_train_set1000 = ...
encoded_train_set3000 = ...

# Set format for PyTorch
encoded_test_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
encoded_train_set1000.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
encoded_train_set3000.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
# Check out how the training sets are encoded
for key, val in encoded_train_set1000[0].items():
    print(f'{key}: {val.size()}')

Example of how your output could look like.

input_ids: torch.Size([???])

attention_mask: torch.Size([???])

labels: torch.Size([???])

📝❓What value should replace the question marks in your print? Should this be the same for all samples? Why/Why not?

## Training

### Training utils

In [ ]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments
import os
os.environ["WANDB_MODE"] = "disabled"

**TODO: Complete the following, reusable functions**

We evaluate on the level of entities, which is the standard for NER: the predicted tags of a sentence are first grouped into entities (for example, `B-PER I-PER` is one `PER` entity), and an entity only counts as correct if both its boundaries and its type match the gold entity. The [seqeval](https://github.com/chakki-works/seqeval) library does this for you. Note that `O` is not an entity type: it only marks words outside entities.

📝❓ Why do we evaluate NER on the level of entities rather than on the level of individual tokens? Hint: what score would a model get on the token level if it predicted `O` for every token? Check the share of `O` tags in your test set.

In [ ]:
from seqeval.metrics import f1_score
import numpy as np


def compute_metrics(preds):
    """
    Compute entity-level macro and micro F1 scores from PredictionOutput

    seqeval expects one list of tag strings (for example "B-PER") per sentence,
    for the gold tags and for the predicted tags.

    Args:
        preds: transformers.trainer_utils.PredictionOutput

    Returns:
        dict with macro_f1 and micro_f1 scores
    """

In [ ]:
def freeze_weights(model):
    """Freeze the weights for a given model.

    Args:
        model: transformers.PreTrainedModel

    Returns:
        model: transformers.PreTrainedModel
    """

### Recommended hyperparameters

You do not need to tune the hyperparameters in this exercise. Please use the following settings, with a batch size of 16:

| Variants | Learning rate | Epochs |
|---|---|---|
| 1 and 2 (whole model fine-tuned) | 5e-5 | 3 |
| 3 and 4 (frozen backbone) | 1e-3 | 10 |

📝❓ The variants with a frozen backbone use a learning rate that is 20 times larger. Why do they need it? Think about which weights are trained in each case and where these weights start from (the load report that transformers prints when you load the model shows which weights are newly initialised).

## Variant 1: 1000 sentences, no frozen weights

**TODO: Initialise your model and set up your training arguments**

Note: if you pass the tokenizer to the `Trainer`, use the argument `processing_class` (the older argument `tokenizer` no longer exists in transformers 5).

**TODO: Train your model ⚡ GPU: 1-2 mins**

**TODO: Compute Metrics/Performance of your model.**

📝❓ Is there a challenge when evaluating the predictions of your model? Why is this challenge present and how do you plan to deal with it?

Hint: Look at the lengths

To avoid rerunning, please also print the metrics of each model that completed training

In [ ]:
# print(metrics)

## Variant 2: 3000 sentences, no frozen weights

In [ ]:
# Repeat after each run to save GPU memory (delete the previous model and trainer first)
if device == 'cuda':
    torch.cuda.empty_cache()

## Variant 3: 1000 sentences, frozen weights

Note for Apple Silicon (MPS): training with a frozen backbone may fail with the error `scaled_dot_product_attention for MPS does not support dropout`. In that case, load the model with the additional argument `attn_implementation="eager"`.

In [ ]:
if device == 'cuda':
    torch.cuda.empty_cache()

## Variant 4: 3000 sentences, frozen weights

In [ ]:
if device == 'cuda':
    torch.cuda.empty_cache()

## Part 1: Results

📝❓ Template:

Summary of the four model variants: entity-level micro and macro F1, and the number of trainable parameters

1. Whole model fine-tuned, 1000 sentences:
2. Whole model fine-tuned, 3000 sentences:
3. Frozen backbone, 1000 sentences:
4. Frozen backbone, 3000 sentences:

📝❓ For one of your variants, also compute the token-level micro and macro F1, including the `O` tag (for example, with `sklearn.metrics.f1_score` on the flattened word-level tags). Compare them with the entity-level micro F1 of the same variant. Why do these scores differ so much?

📝❓ Is it better to freeze or not to freeze the transformer backbone weights? Hypothesize why

# Part 2: Zero-shot NER and anonymisation with GLiNER

We will use a ***[fine-tuned version of GLiNER](https://huggingface.co/urchade/gliner_multi_pii-v1)***, specialized in personally identifiable information (PII).

GLiNER was trained on NER data that is mostly English and fine-tuned on personal data in English, French, German, Spanish, Portuguese and Italian. For other languages, it relies on its multilingual backbone, so its results may be noticeably lower. English entity labels are a safe choice, also for texts in other languages.

⚡ GOAL: Compare zero-shot NER with your fine-tuned models, then create your own small dataset with various PII mentions and anonymise it with GLiNER!

## Load the model

In [ ]:
from gliner import GLiNER

# NOTE: No need to load the model on a GPU: all of Part 2 takes a few minutes on a CPU
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")


In [ ]:
# Code copy-pasted from the model card
text = """
Harilala Rasoanaivo, un homme d'affaires local d'Antananarivo, a enregistré une nouvelle société nommée "Rasoanaivo Enterprises" au Lot II M 92 Antohomadinika. Son numéro est le +261 32 22 345 67, et son adresse électronique est harilala.rasoanaivo@telma.mg. Il a fourni son numéro de sécu 501-02-1234 pour l'enregistrement.
"""

labels = ["work", "booking number", "personally identifiable information", "driver licence", "person", "book", "full address", "company", "actor", "character", "email", "passport number", "Social Security Number", "phone number"]
entities = gliner_model.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

## Zero-shot NER on your test set

GLiNER has never been trained on WikiANN, and you do not train it either: you only give it the names of the entity types. Run it on the same 2,000 test sentences that you used in Part 1 and evaluate it in the same way, so that you can compare it with your four BERT variants.

GLiNER works on raw text and returns character spans, whereas WikiANN is split into words with one IOB tag per word. The helper function below joins the words of a sentence with spaces and converts the spans that GLiNER finds in this text back into IOB tags, so that you can evaluate them with seqeval as in Part 1.

If you run Part 2 in a new runtime (for example, on a CPU), first run the setup cell and the cells of Part 1 that load the data and define `ner_tags`.

In [ ]:
# The entity types of WikiANN, as GLiNER labels
gliner_labels = {"person": "PER", "organization": "ORG", "location": "LOC"}


def spans_to_iob(words, entities):
    """Converts the entities that GLiNER finds in " ".join(words) into one IOB tag per word."""
    starts, position = [], 0
    for word in words:
        starts.append(position)
        position += len(word) + 1
    tags = ["O"] * len(words)
    for entity in sorted(entities, key=lambda e: e["start"]):
        inside = [i for i, start in enumerate(starts)
                  if start < entity["end"] and start + len(words[i]) > entity["start"]]
        for k, i in enumerate(inside):
            if tags[i] == "O":
                tags[i] = ("B-" if k == 0 else "I-") + gliner_labels[entity["label"]]
    return tags


# Example: the first test sentence
words = test_set[0]["tokens"]
entities = gliner_model.predict_entities(" ".join(words), list(gliner_labels), threshold=0.5)
print(words)
print(spans_to_iob(words, entities))

**TODO: Run GLiNER on all test sentences and evaluate it ⚡ CPU: 1-3 mins**

In [ ]:
# TODO: Predict the entities of all test sentences with GLiNER, convert them to IOB tags
# and compute the entity-level micro and macro F1 scores with seqeval, as in Part 1

📝❓ Compare GLiNER with your four BERT variants. Which of them outperform it, and what does this suggest about how much annotated data is needed? When would you rather use GLiNER, and when a fine-tuned model?

📝❓ Look at some sentences where the tags predicted by GLiNER differ from the gold tags. Is it always GLiNER that is wrong? Keep in mind that WikiANN was annotated automatically from the links in Wikipedia articles.

## Anonymise entities

In [ ]:
# TODO: Implement anonymization function
def anonymize_entities(text: str, entities: list[dict]) -> str:
    """Anonymize entities in text by replacing them with tags like <PERSON>, <IBAN> etc.

    Args:
        text (str): The original text
        entities (list): List of entity dictionaries with 'start', 'end', and 'label' keys

    Returns:
        str: Text with entities replaced by the corresponding tags.
    """
    pass

## Your own dataset

Time to create our own, diverse dataset. Make sure to not make it easy for the model (for example, do not mention the label in the sentence, but make sure it can be inferred from the context).

In [ ]:
# TODO: Fill these 2 lists with your own data. pii_texts should be of length (at least) 10.

pii_texts = []

labels = []

In [ ]:
# TODO: Create dataset
dataset: list[dict] = []

Time to see the results!

In [ ]:
for example in dataset:
    print("Original Text:\n", example["text"])
    print("Anonymized Text:\n", example["anonymized_text"])
    print("\n---\n")

## Part 2: Questions

📝❓Discuss the benefits of using GLiNER vs a more traditional NER solution.

📝❓Did you encounter false positives/negatives? Discuss.

---

## Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

### Questions

**Part 1: Data and tokenisation**

- 📝❓ Why do we need separate `B-` and `I-` tags? What could not be represented with only `PER`, `ORG`, `LOC` and `O`?
- 📝❓ Why do you need to be aware of the longest input length within your dataset? Which parameter of the model dictates this?
- 📝❓ The dataset is split into words, and the assigned labels are for words. How should we deal with labels after tokenisation?
- 📝❓ What value should replace the question marks in the printed sizes? Should this be the same for all samples? Why/Why not?

**Part 1: Evaluation and hyperparameters**

- 📝❓ Why do we evaluate NER on the level of entities rather than on the level of individual tokens?
- 📝❓ Why do the variants with a frozen backbone need a larger learning rate?
- 📝❓ Is there a challenge when evaluating the predictions of your model? Why is this challenge present and how do you plan to deal with it?

**Part 1: Results**

- 📝❓ Summarise the four model variants: entity-level micro and macro F1, and the number of trainable parameters.
- 📝❓ Compute the token-level micro and macro F1 (including `O`) for one of your variants and compare them with the entity-level micro F1. Why do these scores differ so much?
- 📝❓ Is it better to freeze or not to freeze the transformer backbone weights? Hypothesise why.

**Part 2: GLiNER**

- 📝❓ Compare GLiNER with your four BERT variants. Which of them outperform it, and what does this suggest about how much annotated data is needed? When would you rather use GLiNER, and when a fine-tuned model?
- 📝❓ Look at some sentences where the tags predicted by GLiNER differ from the gold tags. Is it always GLiNER that is wrong?
- 📝❓ Discuss the benefits of using GLiNER vs a more traditional NER solution.
- 📝❓ Did you encounter false positives/negatives on your own dataset? Discuss.

## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☐ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☐ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used:
